# Chạy AGQO với Qwen3 trên Modal Notebook

1. Chọn GPU L4 hoặc A10 trong cấu hình notebook rồi khởi động lại kernel.
2. Chạy các ô để lấy mã nguồn và cài phần phụ thuộc Qwen3.
3. Tải `modal_upload.zip` vào `/mnt/agqo/input/` hoặc tải riêng `ohlcv_daily.csv` và `members.csv`; manifest là tùy chọn.
4. Chạy các ô từ trên xuống. JSON, báo cáo tiếng Việt, quyết định và hình nằm trong `/mnt/agqo/results/final/`.

Notebook chạy Qwen3-8B NF4 4-bit ngay trên GPU Modal Notebook; không cần API OpenAI. OHLCV được đọc từ tệp bạn tải lên.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_URL = "https://github.com/jncmtam/UTEL_AQ2026.git"
PROJECT_DIR = Path("/mnt/agqo/project")
PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "main", PROJECT_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
print(f"Mã dự án: {PROJECT_DIR}")

In [ ]:
%pip install -e '/mnt/agqo/project[qwen3]'


In [ ]:
import json
import zipfile
import pandas as pd
import torch
from IPython.display import display

DATA_DIR = Path("/mnt/agqo/input")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_ARCHIVE = DATA_DIR / "modal_upload.zip"
OHLCV_PATH = DATA_DIR / "ohlcv_daily.csv"
MEMBERS_PATH = DATA_DIR / "members.csv"
MANIFEST_PATH = DATA_DIR / "manifest.json"

if DATA_ARCHIVE.exists() and not (OHLCV_PATH.exists() and MEMBERS_PATH.exists()):
    with zipfile.ZipFile(DATA_ARCHIVE) as bundle:
        allowed = {"ohlcv_daily.csv", "members.csv", "manifest.json"}
        names = {Path(name).name for name in bundle.namelist()}
        if not {"ohlcv_daily.csv", "members.csv"}.issubset(names):
            raise ValueError("Gói dữ liệu cần chứa ohlcv_daily.csv và members.csv.")
        for name in bundle.namelist():
            if Path(name).name in allowed and not name.endswith("/"):
                (DATA_DIR / Path(name).name).write_bytes(bundle.read(name))

missing = [str(path) for path in (OHLCV_PATH, MEMBERS_PATH) if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Hãy tải modal_upload.zip hoặc các tệp CSV lên /mnt/agqo/input/ bằng bảng tệp Modal rồi chạy lại ô này: "
        + ", ".join(missing)
    )

bars = pd.read_csv(OHLCV_PATH)
members = pd.read_csv(MEMBERS_PATH)
print(f"Dữ liệu: {len(bars):,} dòng; {bars['ticker'].nunique()} mã; "
      f"{bars['date'].min()} đến {bars['date'].max()}")
print(f"Danh sách VN30: {len(members)} mã")
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    print("Bản kê khai:", manifest.get("dataset_name"), manifest.get("retrieved_at_utc"))
display(bars.head())
display(members.head())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có")

## Qwen3-8B, prompt tiếng Anh và tri thức truy xuất

Mặc định dùng `Qwen/Qwen3-8B` lượng tử hóa NF4 4-bit trên GPU L4 hoặc A10. Toàn bộ prompt điều khiển viết bằng tiếng Anh. Vì không giả định Qwen có kiến thức tài chính chuyên sâu, mỗi tín hiệu kèm các định nghĩa tài chính, ý nghĩa trường dữ liệu, hàm mục tiêu, chuẩn exact/regret và giới hạn của dữ liệu trong RAG cục bộ phiên bản hóa.

Tiểu mô hình `StandardScaler` + `LogisticRegression` chỉ ước tính xác suất VN30 ở chế độ biến động cao; nó không dự báo lợi suất. Bộ insight và bộ giám sát áp dụng ngưỡng xác định trên dữ liệu đã đo. Lược đồ trả lời được tạo theo từng tín hiệu và chỉ cho phép `policy_id`, `insight_codes`, `evidence_ids` đã có trong danh sách. Mã Python dựng diễn giải tiếng Việt và cờ rủi ro từ các số đo; Qwen không viết luận cứ tự do hoặc tự đặt tham số.

In [ ]:
# Lưu trọng số trên Volume để lần khởi động sau không phải tải lại.
os.environ["HF_HOME"] = "/mnt/agqo/hf-cache"

MODEL_NAME = "Qwen/Qwen3-8B"
MODEL_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"
REQUIRE_GPU = True
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn GPU L4 hoặc A10 trong cấu hình Modal Notebook rồi khởi động lại nhân.")
print(f"Sẵn sàng tải {MODEL_NAME} tại revision {MODEL_REVISION}; bộ nhớ đệm: {os.environ['HF_HOME']}")
print("Trọng số được nạp một lần và lượng tử hóa NF4 4-bit khi chạy trên CUDA.")


## Chạy walk-forward và QAOA

Lệnh này chạy thí nghiệm đầy đủ. Mỗi tín hiệu chỉ nhận đặc trưng tại ngày đó và kết quả các kỳ đã thoát; Qwen đề xuất cấu hình, bộ giám sát quyết định cấu hình thực thi. 14 cấu hình QAOA chạy đối chứng cố định cùng các bộ giải cổ điển. So sánh N=8/12/16, K=6, p=1/2, mixer X/XY và nhiều bố trí trên cùng chuẩn exact. Lượt đầy đủ có thể mất thời gian do số seed, shots và phép đánh giá mục tiêu. Kết quả xuất gồm báo cáo, JSON, biểu đồ kiểm định và biểu đồ khám phá dữ liệu.

In [ ]:
OUTPUT_DIR = Path("/mnt/agqo/results/final")
FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULT_JSON = OUTPUT_DIR / "agqo_final_walkforward.json"
REPORT_MD = OUTPUT_DIR / "BAO_CAO_AGQO_CUOI_CUNG.md"
GUIDE_CACHE = OUTPUT_DIR / "qwen3_decisions.json"

command = [
    sys.executable,
    str(PROJECT_DIR / "scripts/run_ai_qaoa_walkforward.py"),
    "--data", str(OHLCV_PATH),
    "--members", str(MEMBERS_PATH),
    "--json", str(RESULT_JSON),
    "--report", str(REPORT_MD),
    "--figures", str(FIGURES_DIR),
    "--guide-cache", str(GUIDE_CACHE),
    "--llm-guide", "qwen3",
    "--qwen3-model", MODEL_NAME,
    "--qwen3-revision", MODEL_REVISION,
]
print("Đang chạy:", " ".join(command))
env = os.environ.copy()
env["HF_HOME"] = "/mnt/agqo/hf-cache"
subprocess.run(command, cwd=PROJECT_DIR, env=env, check=True)


## Xem kết quả và insight

In [ ]:
result = json.loads(RESULT_JSON.read_text(encoding="utf-8"))
performance = result["portfolio"]["performance"]
selected_names = [
    name for name in performance
    if name == "AGQO Qwen→giám sát→QAOA / chính sách thích nghi"
    or name == "Bộ chọn quy tắc→QAOA / đối chứng không LLM"
    or name in {"VN30", "VN-Index"}
    or "Ledoit–Wolf" in name
    or "AI + QAOA p=1, bộ trộn X" in name
    or "AI + QAOA p=1, bộ trộn XY" in name
]
rows = []
for name in selected_names:
    metrics = performance[name]["metrics"]
    rows.append({
        "Phương pháp": name,
        "CAGR ròng": metrics.get("cagr"),
        "Sharpe": metrics.get("sharpe"),
        "Sortino": metrics.get("sortino"),
        "Sụt giảm tối đa": metrics.get("max_drawdown"),
        "CVaR tháng 95%": metrics.get("monthly_cvar_95"),
        "Số kỳ": metrics.get("months"),
    })
display(pd.DataFrame(rows).set_index("Phương pháp").style.format(precision=3))
print("Số đề xuất Qwen3:", result["agqo_guidance"]["decision_count"])
print("Fallback giám sát:", result["agqo_guidance"]["execution_guardrail"]["fallback_count"])
print("Số phản hồi cần sửa cấu trúc:", result["agqo_guidance"]["execution_guardrail"]["repaired_llm_response_count"])
print("Mô hình:", MODEL_NAME)
print("Revision:", sorted({d.get("model_revision") for d in result["agqo_guidance"]["decisions"] if d.get("model_revision")}))
print("Cache quyết định:", GUIDE_CACHE)


In [ ]:
from IPython.display import Image, display

figure_paths = sorted(FIGURES_DIR.rglob("*.png"))
print(f"Có {len(figure_paths)} biểu đồ")
for path in figure_paths:
    print(path.name)
    display(Image(filename=str(path)))

## Tải kết quả về máy

JSON kết quả, thống kê dữ liệu, báo cáo tiếng Việt, quyết định Qwen3 và toàn bộ biểu đồ nằm trong `/mnt/agqo/results/final/`. Ô tiếp theo đóng gói chúng để tải qua bảng Files của Modal; dữ liệu đầu vào không được đưa vào gói.

In [ ]:
import zipfile

ARCHIVE_PATH = OUTPUT_DIR / "agqo_final_results.zip"
with zipfile.ZipFile(ARCHIVE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (RESULT_JSON, OUTPUT_DIR / "dataset_summary.json", REPORT_MD, GUIDE_CACHE):
        if path.exists():
            archive.write(path, arcname=path.name)
    for path in figure_paths:
        archive.write(path, arcname=path.relative_to(FIGURES_DIR).as_posix())
print(f"Gói kết quả: {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 1024 / 1024:.1f} MiB)")
print("Mở bảng Files của Modal, vào /mnt/agqo/results/final/ và tải agqo_final_results.zip")
